In [16]:
import pandas as pd

In [17]:
netflix_tv_show_df = pd.read_csv("./titles.csv")
netflix_tv_show_df = netflix_tv_show_df.dropna(axis=0)
netflix_tv_show_df = netflix_tv_show_df.rename({"tmdb_popularity": "imdb_popularity"}, axis=1)
netflix_tv_show_df.drop(["id", "description", "production_countries", "imdb_id", "tmdb_score"], axis=1, inplace=True)
netflix_tv_show_df.head()

,title,type,release_year,age_certification,runtime,genres,seasons,imdb_score,imdb_votes,imdb_popularity
5,Monty Python's Flying Circus,SHOW,1969,TV-14,30,"['comedy', 'european']",4.0,8.8,72895.0,12.919
29,Monty Python's Fliegender Zirkus,SHOW,1972,TV-MA,43,['comedy'],1.0,8.1,2144.0,1.490
47,Seinfeld,SHOW,1989,TV-PG,24,['comedy'],9.0,8.9,302700.0,128.743
55,Knight Rider,SHOW,1982,TV-PG,51,"['action', 'scifi', 'crime', 'drama']",4.0,6.9,33760.0,44.378
57,Thomas & Friends,SHOW,1984,TV-Y,10,"['family', 'comedy', 'music', 'action', 'anima...",24.0,6.5,4948.0,49.384


In [18]:
netflix_tv_show_df = netflix_tv_show_df[netflix_tv_show_df.type != "MOVIE"]
netflix_tv_show_df.drop("type", axis=1, inplace=True)
netflix_tv_show_df.head()

,title,release_year,age_certification,runtime,genres,seasons,imdb_score,imdb_votes,imdb_popularity
5,Monty Python's Flying Circus,1969,TV-14,30,"['comedy', 'european']",4.0,8.8,72895.0,12.919
29,Monty Python's Fliegender Zirkus,1972,TV-MA,43,['comedy'],1.0,8.1,2144.0,1.490
47,Seinfeld,1989,TV-PG,24,['comedy'],9.0,8.9,302700.0,128.743
55,Knight Rider,1982,TV-PG,51,"['action', 'scifi', 'crime', 'drama']",4.0,6.9,33760.0,44.378
57,Thomas & Friends,1984,TV-Y,10,"['family', 'comedy', 'music', 'action', 'anima...",24.0,6.5,4948.0,49.384


In [19]:
netflix_tv_show_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1597 entries, 5 to 5796
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              1597 non-null   object 
 1   release_year       1597 non-null   int64  
 2   age_certification  1597 non-null   object 
 3   runtime            1597 non-null   int64  
 4   genres             1597 non-null   object 
 5   seasons            1597 non-null   float64
 6   imdb_score         1597 non-null   float64
 7   imdb_votes         1597 non-null   float64
 8   imdb_popularity    1597 non-null   float64
dtypes: float64(4), int64(2), object(3)
memory usage: 124.8+ KB


In [25]:
# One Hot Encode And Flatten the Genres
import ast
netflix_tv_show_df['genres'] = netflix_tv_show_df['genres'].apply(ast.literal_eval)
genre_df = netflix_tv_show_df['genres'].apply(lambda x: pd.Series({genre:1 for genre in x}))
genre_df = genre_df.fillna(0).astype(int)

genre_df.head(50)



,[,',c,o,m,e,d,y,",",,...,a,n,],t,i,s,f,l,w,h
5,1,1,1,1,1,1,1,1,1,1,...,1,1,1,0,0,0,0,0,0,0
29,1,1,1,1,1,1,1,1,0,0,...,0,0,1,0,0,0,0,0,0,0
47,1,1,1,1,1,1,1,1,0,0,...,0,0,1,0,0,0,0,0,0,0
55,1,1,1,1,1,1,1,0,1,1,...,1,1,1,1,1,1,1,0,0,0
57,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,0,0
60,1,1,1,1,1,1,1,1,1,1,...,1,1,1,0,1,0,1,1,0,0
64,1,1,0,0,1,0,0,1,0,0,...,1,0,1,0,1,0,1,1,0,0
65,1,1,1,1,1,1,1,1,1,1,...,1,0,1,0,1,0,1,1,0,0
66,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,0,1,1,0,0
67,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,0,0


In [22]:
def set_col_mapping(df, col_type, ignore_list=[]):
    object_columns = df.select_dtypes(include=[col_type]).columns.tolist()
    for col in object_columns:
        if col in ignore_list:
            continue
        int_col_name = col + "_val"
        res = {}
        for x, value in enumerate(df[col].value_counts().index.tolist()):
            res[value] = x
        df[int_col_name] = df[col].map(res)

    return df

In [6]:
netflix_tv_show_remapped_df = set_col_mapping(netflix_tv_show_df, 'object', ["title"])

In [ ]:
netflix_tv_show_remapped_df.head()

In [8]:
netflix_tv_show_remapped_df.to_csv("./cleaned_df.csv", index=False)

In [ ]:
genres = pd.DataFrame(netflix_tv_show_remapped_df.groupby("genres_val")["imdb_score"].mean())
genres.describe()

In [ ]:
genres.head()

In [ ]:
genres['count'] = pd.DataFrame(netflix_tv_show_df.groupby("genres")["title"].count())
genres.head()

In [ ]:
# One Hot Encode the genres
# Closest Integer for Imdb_Score
# 